In [12]:
## Import necessary modules
import time
import numpy as np
import psi4

def diag_lps(diag, A, nel):
    ## Perform the diagonalization and build density matrix
    ## Almost identical to the one used in HF or KS DFT, except 
    ## the density matrix is built from the lowest eigenvector only,
    ## which is normalized to the number of electrons.
    Fp = psi4.core.triplet(A, diag, A, True, False, True)
    nbf = A.shape[0]
    Cp = psi4.core.Matrix(nbf, nbf)
    eigvals = psi4.core.Vector(nbf)
    Fp.diagonalize(Cp, eigvals, psi4.core.DiagonalizeOrder.Ascending)
    C = psi4.core.doublet(A, Cp, False, False)
    Cocc = psi4.core.Matrix(nbf, 1) 
    ## C.np[:, :1] is the lowest eigenvector.
    ## np.sqrt(nel) is the normalization factor.
    ## nel = number of electrons.
    Cocc.np[:] = np.sqrt(nel) * C.np[:, :1]
    D = psi4.core.doublet(Cocc, Cocc, False, True) 
    ## return density matrix and lowest eigenvalue (mu). 
    return D, eigvals.np[0]

def Vpot_init(build_superfunctional, wfn, alias, vname, restricted=True):
    ## Initialize the potential object.
    ## alias is the functional name.
    ## vname is either "RV" for restricted or "UV" for unrestricted.
    sup = build_superfunctional(alias, restricted)[0]
    sup.set_deriv(1)
    sup.allocate()
    Vpot = psi4.core.VBase.build(wfn.basisset(), sup, vname)
    return Vpot

def Vpot_builder(Vpot, D, V, D_half):
    ## Build and calculate a potential on the grid.
    ## D_half is needed to pass a density matrix 
    ## scaled by 0.5 because this is how compute_V works.
    D_half.copy(D)
    D_half.scale(0.5)
    Vpot.set_D([ D_half ])
    Vpot.compute_V([ V ])
    e = Vpot.quadrature_values()['FUNCTIONAL']
    ## Return an energy and the potential.
    return e, V

def lps_solver(maxiter, Pauli, XC, lam, mol, damp, FA, READ=True):
    
    ## Convergence thresholds.
    E_conv = 1.0e-5
    D_conv = 1.0e-5
    
    wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option("BASIS"))
    mints = psi4.core.MintsHelper(wfn.basisset())
    
    ## Number of basis functions.
    nbf = wfn.nso()
    ## Number of electrons.
    nel = wfn.nalpha() + wfn.nbeta()

    print('Number of basis functions:   %d' % nbf)

    build_superfunctional = psi4.driver.dft.build_superfunctional
    ## Initialize a matrix once and rewrite its content every SCF cycle.
    D_half = psi4.core.Matrix(nbf, nbf)

    ## Initialize the Pauli (P) potential.
    VPpot = Vpot_init(build_superfunctional, wfn, Pauli, "RV", restricted=True)
    VPpot.initialize()
    VP_null = psi4.core.Matrix(nbf, nbf)

    ## Initialize the exchange-correlation (XC) potential.
    VXCpot = Vpot_init(build_superfunctional, wfn, XC, "RV", restricted=True)
    VXCpot.initialize()
    VXC_null = psi4.core.Matrix(nbf, nbf)

    ## Initialize the von Weizsacker potential.
    vW = {
        "name": "vW",
        "x_functionals": {"LDA_X": {"alpha": 0.00}},
        "c_functionals": {"GGA_K_VW": {"alpha": 1.00}}
    }
    VvWpot = Vpot_init(build_superfunctional, wfn, vW, "RV", restricted=True)
    VvWpot.initialize()
    VvW_null = psi4.core.Matrix(nbf, nbf)

    ## Calculate and store V, T, H_core, ERI (I), diagonalization matrix (A).
    V = mints.ao_potential()
    T = mints.ao_kinetic()
    H = T.clone()
    H.add(V)
    I = np.asarray(mints.ao_eri())
    A = mints.ao_overlap()
    A.power(-0.5, 1.e-14)
    ## Initialize a Fock matrix, J matrix, and V_G matrix calculated on the grid.
    F = psi4.core.Matrix(nbf, nbf)
    J = psi4.core.Matrix(nbf, nbf)
    VG = psi4.core.Matrix(nbf, nbf)
    D_diff = psi4.core.Matrix(nbf, nbf)
    
    ## Calculate an initial Core guess.
    D, mu = diag_lps(H, A, nel)
    ## Use D_GUESS as an initial guess.
    if READ:
        D = D_GUESS
    
    Enuc = mol.nuclear_repulsion_energy()
    Eold = 0.0
    
    print('\nStarting SCF iterations:')
    t = time.time()
   
    print("\n    Iter            Energy            ChemPot            Delta E         dRMS\n")
    for SCF_ITER in range(1, maxiter + 1):
    
        ## Save current D to D_old for convergence check and Damping.
        D_old = D
        
        ## Calculate two-electron matrix and add to the Fock matrix.
        J_np = np.einsum('pqrs,rs->pq', I, D.np, optimize=True)
        J.np[:] = J_np
        F.copy(H)
        F.axpy(1.0, J)
        if FA[0]:
            if nel == 0:
                F.axpy(0.0, J)
            else: 
                F.axpy(-FA[1]/nel, J)

        ## Calculate all approximate energies and potentials.
        pau_e, VP = Vpot_builder(VPpot, D, VP_null, D_half)
        xc_e, VXC = Vpot_builder(VXCpot, D, VXC_null, D_half)
        vw_e, VvW = Vpot_builder(VvWpot, D, VvW_null, D_half)

        ## Build V_G potential as V_G = V_P + V_xc + (lam - 1)V_vW.
        ## When lam = 1, the total energy includes full vW energy calculated 
        ## as T.vector_dot(D). In KS DFT, T.vector_dot(D) = T_s, but for 
        ## one-orbital systems T_s = T_vW. When lam = 0, the V_vW cancels out
        ## T.vector_dot(D) and the only kinetic contribution is V_P.
        VG.copy(VP)
        VG.axpy(1.0, VXC)
        VG.axpy((lam - 1.0), VvW)

        ## Caclulate G[n] = T_P[n] + E_xc[n] + (lam - 1)T_vW
        g_e = pau_e + xc_e + ( lam - 1.0 ) * vw_e 

        ## E = T_vW + V_ext
        SCF_E = H.vector_dot(D)
        ## E = T_vW + V_ext + J
        SCF_E += 0.5 * J.vector_dot(D)
        if FA[0]:
            SCF_E += 0.5 * J.vector_dot(D) * ( -FA[1]/nel )
        ## E = T_vW + V_ext + J + G
        SCF_E += g_e
        ## E = T_vW + V_ext + J + G + V_NN
        SCF_E += Enuc

        ## Build full Fock matrix and diagonalize.
        F.axpy(1.0, VG)
        D, mu = diag_lps(F, A, nel)
        
        ## Calculate D_diff 
        D_diff.copy(D)
        D_diff.subtract(D_old)
        dRMS = D_diff.rms()

        ## Dynamic damping
        if (dRMS > damp[2]) or (SCF_ITER <= 2):
            current_damp = damp[0]
        else:
            current_damp = damp[1]
        D.scale(1.0 - current_damp)
        D.axpy(current_damp, D_old)

        print('SCF Iter%3d: % 18.8f   % 1.5E   % 1.5E   % 1.5E'
              % (SCF_ITER, SCF_E, mu, (SCF_E - Eold), dRMS))
        
        if (abs(SCF_E - Eold) < E_conv and dRMS < D_conv):
            break
    
        Eold = SCF_E
        
        if SCF_ITER == maxiter:
            SCF_D = D
            print("\nWARNING ! SCF did not converge. The final values are printed")
            return SCF_E, SCF_D, SCF_ITER
    
    SCF_D = D
    
    print('\nTotal time for SCF iterations: %.3f seconds ' % (time.time() - t))

    return SCF_E, SCF_D, SCF_ITER

In [22]:
psi4.core.clean_options()
psi4.core.clean()
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 1
He
symmetry c1
""")
psi4.set_options({'basis': 'Chan2001', 
                 'DFT_SPHERICAL_POINTS': 6,
                  'DFT_RADIAL_POINTS': 1000})

Pauli = {
    "name": "Pauli",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"LDA_K_TF": {"alpha": 1.00}}
}
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}

## Damping factor before cutoff, after cutoff, the cutoff.
damp = [0.8, 0.0, 0.0001]
## Calculate Fermi–Amaldi? Scaling factor.
FA = [False, 1.0]

SCF_E, D, SCF_ITER = lps_solver(2000, Pauli, XC, 1.0, mol, damp, FA, READ=False)
print('\nFinal SCF energy: %.4f hartree' % SCF_E)

Number of basis functions:   19

Starting SCF iterations:

    Iter            Energy            ChemPot            Delta E         dRMS

SCF Iter  1:         1.09975190    1.40907E-02    1.09975E+00    2.02133E-01
SCF Iter  2:         0.11797503   -4.11366E-02   -9.81777E-01    6.35009E-02
SCF Iter  3:        -0.45238574   -8.66912E-02   -5.70361E-01    6.27556E-02
SCF Iter  4:        -0.78552145   -1.21617E-01   -3.33136E-01    5.08276E-02
SCF Iter  5:        -0.98495683   -1.47518E-01   -1.99435E-01    4.06288E-02
SCF Iter  6:        -1.11155216   -1.60927E-01   -1.26595E-01    3.22003E-02
SCF Iter  7:        -1.19795569   -1.61256E-01   -8.64035E-02    2.51472E-02
SCF Iter  8:        -1.26035884   -1.53403E-01   -6.24031E-02    1.93349E-02
SCF Iter  9:        -1.30705228   -1.43222E-01   -4.66934E-02    1.47706E-02
SCF Iter 10:        -1.34278415   -1.34105E-01   -3.57319E-02    1.13794E-02
SCF Iter 11:        -1.37054588   -1.27035E-01   -2.77617E-02    8.92843E-03
SCF Iter 12:   